In [1]:
import datetime
import pandas as pd
import numpy as np
from scipy.io import loadmat
from sklearn.preprocessing import MinMaxScaler

def load_data(battery):
    mat = loadmat('../datasets/battery_data/' + battery + '.mat')
    print('Total data in dataset: ', len(mat[battery][0, 0]['cycle'][0]))
    counter = 0
    dataset = []
    capacity_data = []
    
    for i in range(len(mat[battery][0, 0]['cycle'][0])):
        row = mat[battery][0, 0]['cycle'][0, i]
        if row['type'][0] == 'discharge':
            ambient_temperature = float(row['ambient_temperature'][0][0][0,0])
            date_time = datetime.datetime(int(row['time'][0][0][0,0]),
                                     int(row['time'][0][0][0,1]),
                                     int(row['time'][0][0][0,2]),
                                     int(row['time'][0][0][0,3]),
                                     int(row['time'][0][0][0,4])) + datetime.timedelta(seconds=int(row['time'][0][0][0,5]))
            data = row['data']
            capacity = float(data[0][0]['Capacity'][0][0][0,0])
            for j in range(len(data[0][0]['Voltage_measured'][0][0][0])):
                voltage_measured = float(data[0][0]['Voltage_measured'][0][0][0, j])
                current_measured = float(data[0][0]['Current_measured'][0][0][0, j])
                temperature_measured = float(data[0][0]['Temperature_measured'][0][0][0, j])
                current_load = float(data[0][0]['Current_load'][0][0][0, j])
                voltage_load = float(data[0][0]['Voltage_load'][0][0][0, j])
                time = float(data[0][0]['Time'][0][0][0, j])
                dataset.append([counter + 1, ambient_temperature, date_time, capacity,
                              voltage_measured, current_measured,
                              temperature_measured, current_load,
                              voltage_load, time])
            capacity_data.append([counter + 1, ambient_temperature, date_time, capacity])
            counter = counter + 1
    
    return [pd.DataFrame(data=dataset,
                         columns=['cycle', 'ambient_temperature', 'datetime',
                                  'capacity', 'voltage_measured',
                                  'current_measured', 'temperature_measured',
                                  'current_load', 'voltage_load', 'time']),
            pd.DataFrame(data=capacity_data,
                         columns=['cycle', 'ambient_temperature', 'datetime',
                                  'capacity'])]

# Load data
dataset, capacity_data = load_data('B0005')

# Create SOH
C = dataset['capacity'].iloc[0]
soh_values = (dataset['capacity'] / C).values
soh = pd.DataFrame(data=soh_values, columns=['SoH'])

# Extract features
attribs = ['capacity', 'voltage_measured', 'current_measured',
           'temperature_measured', 'current_load', 'voltage_load', 'time']
train_dataset = dataset[attribs].values

# Apply MinMaxScaler
sc = MinMaxScaler(feature_range=(0, 1))
train_dataset_scaled = sc.fit_transform(train_dataset)

print(f"Scaled dataset shape: {train_dataset_scaled.shape}")
print(f"SOH shape: {soh.shape}")


Total data in dataset:  168
Scaled dataset shape: (50285, 7)
SOH shape: (50285, 1)
